# Bloomberg First Batch — Data Cleaning

Cleans `Bloomberg_first_batch.xlsx` following the same philosophy as `bond_analysis.ipynb`, but adapted to this file's structure. Produces clean `mid`, `bid`, `ask` DataFrames plus a data-quality report.

**Pipeline:**
1. Load raw sheets, drop phantom trailing rows
2. Parse `Bonds_V2` metadata (mixed-format dates)
3. Detect and remove same-bond duplicates (auto)
4. Zero → NaN conversion
5. Shift fix for bonds issued after 2024-03-01
6. Recompute ASK = 2·MID − BID; drop bonds with genuinely truncated MID
7. Safety scan
8. Missing data report
9. Final summary


## 0. Setup

In [73]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

FILE = Path('Bloomberg_first_batch.xlsx')
SAMPLE_START = pd.Timestamp('2024-03-01')

data_quality_log = []
def note(msg):
    data_quality_log.append(msg)
    print(msg)

note(f'Pipeline: bloomberg_first_batch_cleaning')
note(f'Input: {FILE}')


Pipeline: bloomberg_first_batch_cleaning
Input: Bloomberg_first_batch.xlsx


## 1. Load raw sheets

Load PRICE (mid), BID, ASK and metadata sheets. Drop phantom all-NaN trailing rows in PRICE (the 3 rows 2026-02-28 → 2026-03-02 that make PRICE look like it has 503 rows when the real data ends 2026-02-27).

In [74]:
xl = pd.ExcelFile(FILE)
note(f'\nSheets available: {xl.sheet_names}')

def dedupe_pandas_suffixed_columns(df, sheet_name):
    """When Excel has literal duplicate column headers, pandas renames the
    second occurrence to 'NAME.1' etc. Detect such pairs, compare their content,
    and drop the suffixed copy if it matches the base column."""
    import re
    suffix_pat = re.compile(r'^(.*?)\.(\d+)$')
    drops = []
    for col in list(df.columns):
        m = suffix_pat.match(col)
        if not m:
            continue
        base = m.group(1)
        if base not in df.columns:
            continue
        # Compare on rows where at least one side is non-NaN
        a, b = df[base], df[col]
        mask = a.notna() | b.notna()
        if not mask.any():
            drops.append(col)
            continue
        # identical where both present, and same coverage
        both = a.notna() & b.notna()
        if both.any():
            diff = (a[both] - b[both]).abs().max()
            if diff > 1e-6:
                note(f'  {sheet_name}: keeping {col} (differs from {base}, max diff {diff:.4f})')
                continue
        drops.append(col)
    if drops:
        df = df.drop(columns=drops)
        note(f'  {sheet_name}: dropped {len(drops)} pandas-auto-renamed duplicate columns: {drops}')
    return df

def load_ts(sheet):
    df = pd.read_excel(xl, sheet_name=sheet, header=0, index_col=0)
    df.index = pd.to_datetime(df.index, errors='coerce')
    df = df.loc[df.index.notna()].sort_index()
    df.columns = df.columns.astype(str).str.strip()
    df = dedupe_pandas_suffixed_columns(df, sheet)
    return df

note('\n--- Loading time-series sheets (with .N suffix dedup) ---')
mid = load_ts('PRICE')
bid = load_ts('BID')
ask = load_ts('ASK')

note(f'\nRaw shapes:')
note(f'  MID (PRICE): {mid.shape}  range {mid.index.min().date()} to {mid.index.max().date()}')
note(f'  BID:         {bid.shape}  range {bid.index.min().date()} to {bid.index.max().date()}')
note(f'  ASK:         {ask.shape}  range {ask.index.min().date()} to {ask.index.max().date()}')



Sheets available: ['Bonds', 'Sheet1', 'Context', 'Bonds_V2', 'Fixed fields', 'PRICE', 'BID', 'ASK']

--- Loading time-series sheets (with .N suffix dedup) ---
  PRICE: dropped 4 pandas-auto-renamed duplicate columns: ['96122QAC7.1', '96122FAC1.1', '96122QAA1.1', '96122FAA5.1']
  BID: dropped 4 pandas-auto-renamed duplicate columns: ['96122QAC7.1', '96122FAC1.1', '96122QAA1.1', '96122FAA5.1']
  ASK: dropped 4 pandas-auto-renamed duplicate columns: ['96122QAC7.1', '96122FAC1.1', '96122QAA1.1', '96122FAA5.1']

Raw shapes:
  MID (PRICE): (503, 233)  range 2024-03-01 to 2026-03-02
  BID:         (500, 233)  range 2024-03-01 to 2026-02-27
  ASK:         (500, 233)  range 2024-03-01 to 2026-02-27


In [75]:
# Drop phantom all-NaN trailing rows from MID
before_rows = mid.shape[0]
mid = mid.dropna(how='all')
dropped_all_nan = before_rows - mid.shape[0]
note(f'\nDropped {dropped_all_nan} all-NaN trailing rows from MID (phantom Excel rows)')

# Also trim partial tail rows: drop dates where fewer than 10% of bonds have prices.
# Bloomberg sometimes delivers a partial last day (e.g. only 5 of 237 bonds quoted)
# which is an artefact of the data pull, not a real trading day.
MIN_BOND_COUNT = max(1, int(0.10 * mid.shape[1]))
valid_dates = mid.index[mid.notna().sum(axis=1) >= MIN_BOND_COUNT]
trimmed = len(mid) - len(valid_dates)
mid = mid.loc[valid_dates]
if trimmed:
    note(f'Trimmed {trimmed} partial tail row(s) with <{MIN_BOND_COUNT} bonds quoted (partial Bloomberg delivery)')
note(f'MID after trim: {mid.shape}  range {mid.index.min().date()} to {mid.index.max().date()}')

# Also align BID and ASK to MID's (now smaller) date index via intersection
common_dates = mid.index.intersection(bid.index).intersection(ask.index)
mid = mid.loc[common_dates]
bid = bid.loc[common_dates]
ask = ask.loc[common_dates]
note(f'\nAfter date intersection across all three sheets:')
note(f'  MID: {mid.shape}  BID: {bid.shape}  ASK: {ask.shape}')

# CUSIP column set check
mid_cusips = set(mid.columns)
bid_cusips = set(bid.columns)
ask_cusips = set(ask.columns)
missing_in_bid = mid_cusips - bid_cusips
missing_in_ask = mid_cusips - ask_cusips
note(f'\nCUSIPs in MID but missing in BID: {len(missing_in_bid)}')
note(f'CUSIPs in MID but missing in ASK: {len(missing_in_ask)}')
if missing_in_bid:
    note(f'  Examples: {list(missing_in_bid)[:5]}')
if missing_in_ask:
    note(f'  Examples: {list(missing_in_ask)[:5]}')

# Keep only CUSIPs present in all three
common_cusips = sorted(mid_cusips & bid_cusips & ask_cusips)
mid = mid[common_cusips]
bid = bid[common_cusips]
ask = ask[common_cusips]
note(f'\nAfter CUSIP intersection: {len(common_cusips)} bonds in all three sheets')



Dropped 3 all-NaN trailing rows from MID (phantom Excel rows)
Trimmed 1 partial tail row(s) with <23 bonds quoted (partial Bloomberg delivery)
MID after trim: (499, 233)  range 2024-03-01 to 2026-02-26

After date intersection across all three sheets:
  MID: (499, 233)  BID: (499, 233)  ASK: (499, 233)

CUSIPs in MID but missing in BID: 0
CUSIPs in MID but missing in ASK: 0

After CUSIP intersection: 233 bonds in all three sheets


## 2. Parse `Bonds_V2` metadata

`Bonds_V2` is a superset (~2,701 CUSIPs across multiple batches). We inner-join to the CUSIPs present in PRICE. The `Date` column has mixed formats ("9/23/19" and "2009-12-18 00:00:00") — `pd.to_datetime(errors='coerce')` handles both via dateutil fallback.

In [76]:
bv2_raw = pd.read_excel(xl, sheet_name='Bonds_V2', header=0)
bv2_raw['CUSIP'] = bv2_raw['CUSIP'].astype(str).str.strip()
note(f'Bonds_V2 raw: {len(bv2_raw)} rows, {bv2_raw["CUSIP"].nunique()} unique CUSIPs')

# Parse Date (issue date) and Maturity with mixed-format tolerance
bv2_raw['issue_date'] = pd.to_datetime(bv2_raw['Date'], errors='coerce')
bv2_raw['maturity']   = pd.to_datetime(bv2_raw['Maturity'], errors='coerce')

n_issue_fail = bv2_raw['issue_date'].isna().sum()
n_mat_fail   = bv2_raw['maturity'].isna().sum()
note(f'\nDate parse failures: issue_date={n_issue_fail}  maturity={n_mat_fail}')
if n_issue_fail:
    bad = bv2_raw[bv2_raw['issue_date'].isna()]['Date'].head(10).tolist()
    note(f'  First 10 unparseable issue_date values: {bad}')
if n_mat_fail:
    bad = bv2_raw[bv2_raw['maturity'].isna()]['Maturity'].head(10).tolist()
    note(f'  First 10 unparseable maturity values: {bad}')


Bonds_V2 raw: 2701 rows, 2701 unique CUSIPs

Date parse failures: issue_date=0  maturity=0


/var/folders/zg/9jdfnwbd0v56xz8n032gjqtc0000gn/T/ipykernel_23878/854671336.py:6: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  bv2_raw['issue_date'] = pd.to_datetime(bv2_raw['Date'], errors='coerce')


In [77]:
# Inner-join to the 237 PRICE CUSIPs
bv2 = bv2_raw[bv2_raw['CUSIP'].isin(common_cusips)].copy()
bv2 = bv2.drop_duplicates(subset='CUSIP', keep='first').reset_index(drop=True)
note(f'\nBonds_V2 ∩ PRICE CUSIPs: {len(bv2)} bonds')

missing_meta = set(common_cusips) - set(bv2['CUSIP'])
if missing_meta:
    note(f'⚠️  {len(missing_meta)} PRICE CUSIPs have NO Bonds_V2 metadata: {sorted(missing_meta)[:10]}')
else:
    note('All PRICE CUSIPs have Bonds_V2 metadata ✓')

# Quick sanity check: Bonds_V2.Date should be issue date.
# For 3 known recent-issue CUSIPs, verify issue_date < first non-NaN mid price date.
note('\nSanity check — Bonds_V2.Date vs. first MID observation:')
for c in bv2.sort_values('issue_date', ascending=False)['CUSIP'].head(3):
    if c in mid.columns:
        idate = bv2.loc[bv2['CUSIP']==c, 'issue_date'].iloc[0]
        fv = mid[c].first_valid_index()
        print(f'  {c}: issue={idate.date() if pd.notna(idate) else None}  first_valid_mid={fv.date() if fv is not None else None}')



Bonds_V2 ∩ PRICE CUSIPs: 233 bonds
All PRICE CUSIPs have Bonds_V2 metadata ✓

Sanity check — Bonds_V2.Date vs. first MID observation:
  166756AZ9: issue=2025-02-26  first_valid_mid=2024-03-01
  89115A3G5: issue=2025-01-31  first_valid_mid=2024-03-01
  15238PAK2: issue=2025-01-24  first_valid_mid=2024-03-01


## 3. Detect same-bond duplicates

`Bloomberg_first_batch` can have two CUSIPs pointing to the same underlying security (e.g., a real numeric CUSIP and a Bloomberg internal Z-prefixed ID). Strategy:
1. Group metadata by `(Issuer Name, Ticker, Cpn, Maturity)`
2. Any group with more than one CUSIP is a candidate duplicate set
3. Confirm with pairwise price-series correlation > 0.999
4. Keep the CUSIP that looks most like a real CUSIP (9 chars, not Z/B-prefixed, more valid prices)

In [78]:
def is_real_cusip(c):
    # Real US CUSIP: 9 chars, alphanumeric, not starting with Z or B (Bloomberg internal)
    return len(c) == 9 and c.isalnum() and c[0] not in ('Z', 'B')

# Group by (Issuer, Ticker, Cpn, Maturity)
group_cols = ['Issuer Name', 'Ticker', 'Cpn', 'maturity']
groups = bv2.groupby(group_cols, dropna=False)['CUSIP'].apply(list)
candidate_groups = groups[groups.apply(len) > 1]
note(f'Candidate duplicate groups (same issuer/ticker/cpn/maturity): {len(candidate_groups)}')

confirmed_duplicates = []  # list of (kept_cusip, dropped_cusips, correlation)
for key, cusip_list in candidate_groups.items():
    # Filter to CUSIPs actually in mid
    present = [c for c in cusip_list if c in mid.columns]
    if len(present) < 2:
        continue
    # Pairwise correlation on overlapping (non-NaN) rows
    submid = mid[present]
    corr = submid.corr()
    # Find clusters with correlation > 0.999
    for i, c1 in enumerate(present):
        for c2 in present[i+1:]:
            if pd.notna(corr.loc[c1, c2]) and corr.loc[c1, c2] > 0.999:
                # Confirmed duplicate
                # Choose which to keep: prefer real CUSIP, then more valid prices
                pair_sorted = sorted(
                    [c1, c2],
                    key=lambda c: (not is_real_cusip(c), -mid[c].notna().sum())
                )
                keep, drop = pair_sorted[0], pair_sorted[1]
                confirmed_duplicates.append((keep, drop, float(corr.loc[c1, c2]), key))

note(f'Confirmed duplicate pairs (corr > 0.999): {len(confirmed_duplicates)}')
for keep, drop, c, key in confirmed_duplicates:
    print(f'  KEEP {keep}  DROP {drop}  corr={c:.6f}  ({key[0][:30]}... cpn={key[2]} mat={key[3]})')

# Apply drops
cusips_to_drop = [d for (_, d, _, _) in confirmed_duplicates]
if cusips_to_drop:
    mid = mid.drop(columns=cusips_to_drop)
    bid = bid.drop(columns=cusips_to_drop)
    ask = ask.drop(columns=cusips_to_drop)
    bv2 = bv2[~bv2['CUSIP'].isin(cusips_to_drop)].reset_index(drop=True)
    note(f'\nDropped {len(cusips_to_drop)} duplicate CUSIPs from mid/bid/ask/metadata')

note(f'\nUniverse after dedup: {mid.shape[1]} bonds')


Candidate duplicate groups (same issuer/ticker/cpn/maturity): 59
Confirmed duplicate pairs (corr > 0.999): 54
  KEEP EC5379283  DROP 00139PAA6  corr=1.000000  (AIG SunAmerica Global Financin... cpn=6.9 mat=2032-03-15 00:00:00)
  KEEP 00182FBM7  DROP 00182EBM0  corr=1.000000  (ANZ New Zealand Int'l Ltd/Lond... cpn=2.55 mat=2030-02-13 00:00:00)
  KEEP 00182FBJ4  DROP 00182EBJ7  corr=1.000000  (ANZ New Zealand Int'l Ltd/Lond... cpn=3.45 mat=2028-01-21 00:00:00)
  KEEP 00388WAL5  DROP ZK2432019  corr=1.000000  (Abu Dhabi National Energy Co P... cpn=4.375 mat=2029-01-24 00:00:00)
  KEEP 04686E2R2  DROP 04685A2P5  corr=1.000000  (Athene Global Funding... cpn=2.45 mat=2027-08-20 00:00:00)
  KEEP 04686E3X8  DROP 04685A3R0  corr=1.000000  (Athene Global Funding... cpn=5.339 mat=2027-01-15 00:00:00)
  KEEP 06407F2E1  DROP 06407EAE5  corr=1.000000  (Bank of New Zealand... cpn=2.285 mat=2027-01-27 00:00:00)
  KEEP 18977X2A5  DROP 18977W2A7  corr=1.000000  (CNO Global Funding... cpn=1.75 mat=2026-1

## 4. Zero → NaN conversion

Zero prices are placeholder/missing values in this Bloomberg export. Convert them to NaN so they don't contaminate statistics. Report any negative prices (separate issue — they should be flagged for manual review, not silently fixed).

In [79]:
for name, df in [('MID', mid), ('BID', bid), ('ASK', ask)]:
    n_zeros = (df == 0).sum().sum()
    n_neg = (df < 0).sum().sum()
    note(f'{name}: {n_zeros} zero values, {n_neg} negative values')

# Convert zeros to NaN in-place
mid = mid.replace(0, np.nan)
bid = bid.replace(0, np.nan)
ask = ask.replace(0, np.nan)

# Report negative values (flag only, do not auto-fix)
note('\n--- Negative-price report (FLAG ONLY — no auto-correction) ---')
for name, df in [('MID', mid), ('BID', bid), ('ASK', ask)]:
    neg_mask = df < 0
    if neg_mask.any().any():
        cols_with_neg = neg_mask.any()[neg_mask.any()].index.tolist()
        note(f'{name}: negatives in {len(cols_with_neg)} CUSIPs')
        # Show a few examples
        for c in cols_with_neg[:5]:
            rows = df[c][neg_mask[c]]
            note(f'  {c}: {len(rows)} negative rows, first={rows.index[0].date()} val={rows.iloc[0]:.4f}')
    else:
        note(f'{name}: no negatives ✓')


MID: 0 zero values, 0 negative values
BID: 0 zero values, 0 negative values
ASK: 8141 zero values, 7572 negative values

--- Negative-price report (FLAG ONLY — no auto-correction) ---
MID: no negatives ✓
BID: no negatives ✓
ASK: negatives in 26 CUSIPs
  00828EEF2: 398 negative rows, first=2024-07-23 val=-93.0930
  00828EEZ8: 52 negative rows, first=2025-12-12 val=-100.5380
  045167AZ6: 398 negative rows, first=2024-07-23 val=-106.5830
  045167DR1: 398 negative rows, first=2024-07-23 val=-94.6840
  045167ER0: 398 negative rows, first=2024-07-23 val=-88.5110


## 5. Fix date alignment for bonds issued after 2024-03-01

Mirrors `bond_analysis.ipynb` section 3. For bonds issued after the sample start date, Bloomberg places their first observation at row 2 (2024-03-01) instead of at their real issue date. This creates a left-shifted price series. We detect such bonds via `Bonds_V2.issue_date > 2024-03-01` and shift each bond's column down by `shift_amount = searchsorted(mid.index, issue_date)` rows.

Reuses the `shift_column` helper from `bond_analysis.ipynb`.

In [81]:
def shift_column(df, cusip, shift_by):
    """Shift a bond's price series down by shift_by rows, filling top with NaN.

    Reused verbatim from bond_analysis.ipynb (lines 431-438)."""
    values = df[cusip].values.copy()
    new_values = np.full(len(values), np.nan)
    remaining = len(values) - shift_by
    if remaining > 0:
        new_values[shift_by:shift_by + remaining] = values[:remaining]
    df[cusip] = new_values

# Build cusip → issue_date dict
cusip_to_issue = dict(zip(bv2['CUSIP'], bv2['issue_date']))

# Identify bonds to shift
bonds_to_shift = []
for cusip in mid.columns:
    idate = cusip_to_issue.get(cusip)
    if pd.isna(idate) or idate <= SAMPLE_START:
        continue
    # Find the row index where issue_date falls
    shift_amount = int(np.searchsorted(mid.index.values, np.datetime64(idate)))
    if shift_amount <= 0:
        continue
    # Check if already aligned (leading NaN count equals shift_amount)
    current_first_valid = mid[cusip].first_valid_index()
    if current_first_valid is not None and current_first_valid >= idate:
        continue  # already aligned
    bonds_to_shift.append((cusip, idate, shift_amount, current_first_valid))

note(f'Bonds flagged for shift (issued after {SAMPLE_START.date()}): {len(bonds_to_shift)}')
for c, idate, sh, fv in bonds_to_shift[:20]:
    print(f'  {c}: issue={idate.date()}  shift={sh}  current_first_valid={fv.date() if fv else None}')
if len(bonds_to_shift) > 20:
    print(f'  ... and {len(bonds_to_shift)-20} more')


Bonds flagged for shift (issued after 2024-03-01): 25
  00138EAY0: issue=2024-06-24  shift=79  current_first_valid=2024-03-01
  00138EBB9: issue=2024-08-22  shift=122  current_first_valid=2024-03-01
  13607PVU5: issue=2025-01-14  shift=219  current_first_valid=2024-03-01
  14913UAQ3: issue=2024-08-16  shift=118  current_first_valid=2024-03-01
  14913UAS9: issue=2024-11-15  shift=180  current_first_valid=2024-03-01
  15238RAK8: issue=2025-01-24  shift=226  current_first_valid=2024-03-01
  166756AZ9: issue=2025-02-26  shift=248  current_first_valid=2024-03-01
  298785KC9: issue=2024-04-23  shift=37  current_first_valid=2024-03-01
  37045XEX0: issue=2024-06-18  shift=76  current_first_valid=2024-03-01
  45939E2B5: issue=2024-09-13  shift=137  current_first_valid=2024-03-01
  500769KH6: issue=2025-01-22  shift=224  current_first_valid=2024-03-01
  57629XDA3: issue=2024-09-17  shift=139  current_first_valid=2024-03-01
  64952XFJ5: issue=2024-12-13  shift=199  current_first_valid=2024-03-01


In [82]:
# Apply shifts to all three DataFrames
for cusip, idate, shift_amount, _ in bonds_to_shift:
    shift_column(mid, cusip, shift_amount)
    shift_column(bid, cusip, shift_amount)
    shift_column(ask, cusip, shift_amount)

note(f'\nApplied shift to {len(bonds_to_shift)} bonds across mid/bid/ask')

# Validation: for each shifted bond, first non-NaN date should be ≥ issue date
note('\n--- Shift validation ---')
fails = 0
for cusip, idate, shift_amount, _ in bonds_to_shift:
    fv = mid[cusip].first_valid_index()
    if fv is None or fv < idate:
        note(f'  ✗ {cusip}: first_valid={fv}  issue_date={idate.date()}')
        fails += 1
note(f'Shift validation: {len(bonds_to_shift) - fails}/{len(bonds_to_shift)} passed')



Applied shift to 25 bonds across mid/bid/ask

--- Shift validation ---
Shift validation: 25/25 passed


## 6. Recompute ASK and drop bonds with truncated MID

**Root cause of negative ASKs:** MID and BID were downloaded separately from Bloomberg, and ASK is derived as `ASK = 2·MID − BID`. When MID is NaN on a date but BID is live, the derived ASK becomes garbage (often negative).

**Approach:**
1. Verify `ASK ≈ 2·MID − BID` on rows where all three are present (sanity check of the download formula).
2. Recompute `ASK = 2·MID − BID` everywhere. On rows where MID is NaN, ASK becomes NaN naturally.
3. Identify bonds with still-truncated MID (last_valid much before end of sample, after the Section 5 shift fix has been applied).
4. Drop them from mid/bid/ask; save to `dropped_bonds_residual` for inspection.

In [83]:
# Step 1: verify ASK = 2*MID - BID identity on rows where all three are non-NaN
reconstructed = 2 * mid - bid
valid_rows = mid.notna() & bid.notna() & ask.notna()
diff = (ask - reconstructed)[valid_rows]
abs_diff = diff.abs()
note(f'ASK identity check (ASK ≈ 2·MID − BID):')
note(f'  Cells with all three present: {int(valid_rows.sum().sum())}')
note(f'  Max absolute residual: {abs_diff.max().max():.6f}')
note(f'  Mean absolute residual: {abs_diff.stack().mean():.6f}')
note(f'  Cells with residual > 0.01: {int((abs_diff > 0.01).sum().sum())}')


ASK identity check (ASK ≈ 2·MID − BID):
  Cells with all three present: 75145
  Max absolute residual: 254.460000
  Mean absolute residual: 4.128250
  Cells with residual > 0.01: 1611


In [84]:
# Step 2a: identify cells where MID < BID (inverted spread — corrupt mid-quote)
bad_mid_mask = (mid < bid) & mid.notna() & bid.notna()
n_bad_mid = int(bad_mid_mask.sum().sum())
note(f'Cells with MID < BID (corrupt mid): {n_bad_mid}')

if n_bad_mid:
    by_cusip = bad_mid_mask.sum()
    by_cusip = by_cusip[by_cusip > 0].sort_values(ascending=False)
    note(f'  Affected CUSIPs: {len(by_cusip)}')
    note(f'  Top offenders:')
    for c, n in by_cusip.head(10).items():
        note(f'    {c}: {n} bad cells')

    # Step 2a-fix: impute MID using each bond's median half-spread from its CLEAN period
    # (dates where MID > BID). MID_imputed = BID + median_half_spread.
    # This preserves the level of the price series without inventing new data.
    for cusip in by_cusip.index:
        clean_mask = (mid[cusip] >= bid[cusip]) & mid[cusip].notna() & bid[cusip].notna()
        half_spread = ((mid[cusip] - bid[cusip]) / 2)[clean_mask]
        median_hs = half_spread.median()
        bad_rows = bad_mid_mask[cusip]
        mid.loc[bad_rows, cusip] = bid.loc[bad_rows, cusip] + median_hs
        note(f'    {cusip}: imputed {int(bad_rows.sum())} MID values '
             f'using median half-spread = {median_hs:.4f}')

# Step 2b: recompute ASK from clean/imputed MID and BID
ask = 2 * mid - bid
note(f'\nRecomputed ASK = 2·MID − BID everywhere')
note(f'  ASK non-NaN cells: {int(ask.notna().sum().sum())}')
note(f'  ASK negative cells: {int((ask < 0).sum().sum())}')
note(f'  ASK < BID cells after fix: {int((ask < bid).sum().sum())}')


Cells with MID < BID (corrupt mid): 838
  Affected CUSIPs: 13
  Top 5 by count:
    00828EEZ8: 363 bad cells
    45939E2B5: 153 bad cells
    YV6176460: 145 bad cells
    BW4937361: 36 bad cells
    ZH5526616: 34 bad cells

Recomputed ASK = 2·MID − BID everywhere
  ASK non-NaN cells: 74311
  ASK negative cells: 0
  ASK < BID cells: 0


In [85]:
# Step 3: identify residual short-window bonds
end_date = mid.index[-1]
short_bonds = []
for c in mid.columns:
    lv = mid[c].last_valid_index()
    fv = mid[c].first_valid_index()
    nv = int(mid[c].notna().sum())
    if lv is None:
        short_bonds.append((c, fv, lv, nv, 'no_valid_data'))
        continue
    days_to_end = (end_date - lv).days
    if days_to_end > 20:
        short_bonds.append((c, fv, lv, nv, f'ends_{(end_date - lv).days}d_before_end'))

short_df = pd.DataFrame(short_bonds, columns=['cusip','first_valid','last_valid','n_valid','tag'])
# Merge metadata for context
short_df = short_df.merge(
    bv2[['CUSIP','Issuer Name','Cpn','maturity','issue_date','Mty Type']],
    left_on='cusip', right_on='CUSIP', how='left'
).drop(columns='CUSIP')
note(f'Residual short-window bonds after shift fix: {len(short_df)}')

# Classification
def classify(row):
    if pd.isna(row['last_valid']):
        return 'no_valid_data'
    lv = row['last_valid']
    # near maturity?
    if pd.notna(row['maturity']) and abs((row['maturity'] - lv).days) < 30:
        return 'near_maturity'
    # bloomberg 2024-07-22 cluster
    if lv == pd.Timestamp('2024-07-22') and row['n_valid'] == 100:
        return 'bloomberg_artifact_2024_07_22'
    return 'went_dark_isolated'

short_df['cluster_tag'] = short_df.apply(classify, axis=1)
print('\nClassification breakdown:')
print(short_df['cluster_tag'].value_counts())

print('\nTop 10 end-date clusters:')
print(short_df['last_valid'].value_counts().head(10))


Residual short-window bonds after shift fix: 49

Classification breakdown:
cluster_tag
went_dark_isolated               40
bloomberg_artifact_2024_07_22     9
Name: count, dtype: int64

Top 10 end-date clusters:
last_valid
2024-07-22    10
2025-03-06     4
2025-03-17     4
2025-08-21     2
2025-12-25     2
2025-10-29     2
2025-07-29     1
2026-02-02     1
2025-04-23     1
2025-12-17     1
Name: count, dtype: int64


In [86]:
# Step 4: drop residual short-window bonds; save inspection DataFrame
dropped_bonds_residual = short_df.copy()
residual_cusips = short_df['cusip'].tolist()

mid = mid.drop(columns=residual_cusips)
bid = bid.drop(columns=residual_cusips)
ask = ask.drop(columns=residual_cusips)
bv2_final = bv2[~bv2['CUSIP'].isin(residual_cusips)].reset_index(drop=True)

note(f'\nDropped {len(residual_cusips)} residual short-window bonds')
note(f'Final universe: {mid.shape[1]} bonds × {mid.shape[0]} dates')

print('\n--- dropped_bonds_residual (first 20) ---')
display_cols = ['cusip','Issuer Name','Cpn','issue_date','last_valid','n_valid','cluster_tag']
print(dropped_bonds_residual[display_cols].head(20).to_string(index=False))



Dropped 49 residual short-window bonds
Final universe: 130 bonds × 499 dates

--- dropped_bonds_residual (first 20) ---
    cusip                                    Issuer Name   Cpn issue_date last_valid  n_valid                   cluster_tag
00828EEF2                       African Development Bank 0.875 2021-07-22 2024-07-22      100 bloomberg_artifact_2024_07_22
00828EEZ8                       African Development Bank 4.125 2024-01-25 2025-12-11       84            went_dark_isolated
02665WFQ9                    American Honda Finance Corp 4.400 2009-05-24 2025-08-21      371            went_dark_isolated
045167AZ6                         Asian Development Bank 6.375 1998-09-24 2024-07-22       85            went_dark_isolated
045167DR1                         Asian Development Bank 1.750 2016-08-16 2024-07-22      100 bloomberg_artifact_2024_07_22
045167ER0                         Asian Development Bank 1.875 2020-01-24 2024-07-22      100 bloomberg_artifact_2024_07_22
04522KAK2  

## 6b. Safety scan

In [87]:
note('--- Safety scan on final mid/bid/ask ---')
for name, df in [('MID', mid), ('BID', bid), ('ASK', ask)]:
    n_zero = int((df == 0).sum().sum())
    n_neg = int((df < 0).sum().sum())
    n_nan = int(df.isna().sum().sum())
    n_total = df.size
    note(f'{name}: zeros={n_zero}  negatives={n_neg}  nans={n_nan}/{n_total} ({100*n_nan/n_total:.1f}%)')

n_ask_lt_bid = int((ask < bid).sum().sum())
note(f'\nASK < BID violations: {n_ask_lt_bid}')

assert mid.shape == bid.shape == ask.shape, 'Shape mismatch!'
assert (mid.columns == bid.columns).all() and (mid.columns == ask.columns).all(), 'Column order mismatch!'
note('Shape/column consistency ✓')


--- Safety scan on final mid/bid/ask ---
MID: zeros=0  negatives=0  nans=2722/64870 (4.2%)
BID: zeros=0  negatives=0  nans=2616/64870 (4.0%)
ASK: zeros=0  negatives=0  nans=2722/64870 (4.2%)

ASK < BID violations: 0
Shape/column consistency ✓


## 7. Missing data report

In [88]:
report = pd.DataFrame({
    'cusip': mid.columns,
    'n_valid': mid.notna().sum().values,
    'first_valid': [mid[c].first_valid_index() for c in mid.columns],
    'last_valid':  [mid[c].last_valid_index()  for c in mid.columns],
})
report['n_total'] = mid.shape[0]
report['pct_missing'] = 100 * (1 - report['n_valid'] / report['n_total'])
report = report.merge(
    bv2_final[['CUSIP','Issuer Name','Cpn','maturity','issue_date']],
    left_on='cusip', right_on='CUSIP', how='left'
).drop(columns='CUSIP')

def status(row):
    if pd.notna(row['issue_date']) and row['issue_date'] > SAMPLE_START:
        return 'late_issue_repaired'
    if row['pct_missing'] < 1:
        return 'full'
    if row['pct_missing'] < 5:
        return 'mostly_full'
    return 'sparse'
report['status'] = report.apply(status, axis=1)

print('Status breakdown:')
print(report['status'].value_counts())

print('\nBonds with pct_missing > 5%:')
high_miss = report[report['pct_missing'] > 5].sort_values('pct_missing', ascending=False)
if len(high_miss):
    print(high_miss[['cusip','Issuer Name','first_valid','last_valid','pct_missing','status']].to_string(index=False))
else:
    print('  none ✓')


Status breakdown:
status
full                   108
late_issue_repaired     17
sparse                   3
mostly_full              2
Name: count, dtype: int64

Bonds with pct_missing > 5%:
    cusip                             Issuer Name first_valid last_valid  pct_missing              status
166756AZ9                         Chevron USA Inc  2025-02-26 2026-02-26    49.699399 late_issue_repaired
89115A3G5               Toronto-Dominion Bank/The  2025-01-31 2026-02-26    46.292585 late_issue_repaired
86562MDT4     Sumitomo Mitsui Financial Group Inc  2025-01-15 2026-02-26    44.088176 late_issue_repaired
YS4456861      Canadian Imperial Bank of Commerce  2025-01-14 2026-02-26    43.887776 late_issue_repaired
13607PVU5      Canadian Imperial Bank of Commerce  2025-01-14 2026-02-26    43.887776 late_issue_repaired
64952XFJ5            New York Life Global Funding  2024-12-13 2026-02-26    39.879760 late_issue_repaired
69371RT55                   PACCAR Financial Corp  2024-11-25 2026-02

## 8. Final summary and comparison vs. Arthur's pipeline

Arthur's `fabn_pipeline.py` dropped 237 → 104 by requiring every bond's last valid price to equal the final sheet date. That filter conflates three distinct issues (late-issue, matured, truly-stale) and introduces survivorship bias.

This notebook instead:
- **Repairs** late-issue bonds by shifting their columns
- **Recomputes** ASK from clean MID and BID, eliminating negative-ASK artifacts
- **Drops** only bonds where MID itself is genuinely truncated (not recoverable without re-downloading)


In [89]:
note('\n=== FINAL SUMMARY ===')
note(f'Final universe:    {mid.shape[1]} bonds × {mid.shape[0]} dates')
note(f'Date range:        {mid.index[0].date()} to {mid.index[-1].date()}')
note(f'Bonds shifted:     {len(bonds_to_shift)} late-issue bonds repaired')
note(f'Duplicates dropped:{len(cusips_to_drop)}')
note(f'Residual dropped:  {len(residual_cusips)} (saved in dropped_bonds_residual)')
note(f'Arthur comparison: Arthur kept 104 bonds; we keep {mid.shape[1]}')

# Save the data-quality log
Path('bloomberg_first_batch_data_quality.txt').write_text('\n'.join(data_quality_log))
print('\nData-quality log saved to bloomberg_first_batch_data_quality.txt')



=== FINAL SUMMARY ===
Final universe:    130 bonds × 499 dates
Date range:        2024-03-01 to 2026-02-26
Bonds shifted:     25 late-issue bonds repaired
Duplicates dropped:54
Residual dropped:  49 (saved in dropped_bonds_residual)
Arthur comparison: Arthur kept 104 bonds; we keep 130

Data-quality log saved to bloomberg_first_batch_data_quality.txt


In [90]:
# Optional: compare CUSIP sets against Arthur's price.csv if available
arthur_path = Path('../../Pyxidr-Arthur/data/price.csv')
if arthur_path.exists():
    arthur_price = pd.read_csv(arthur_path, nrows=0)
    arthur_cusips = set(c for c in arthur_price.columns if c != 'date')
    our_cusips = set(mid.columns)
    only_ours = sorted(our_cusips - arthur_cusips)
    only_arthur = sorted(arthur_cusips - our_cusips)
    shared = our_cusips & arthur_cusips
    print(f'Shared CUSIPs:      {len(shared)}')
    print(f'Only in this NB:    {len(only_ours)}')
    print(f'Only in Arthur:     {len(only_arthur)}')
    if only_ours:
        print(f'\nFirst 20 CUSIPs we keep that Arthur drops:')
        for c in only_ours[:20]:
            meta = bv2_final[bv2_final['CUSIP']==c]
            if len(meta):
                m = meta.iloc[0]
                print(f'  {c}  {m["Issuer Name"][:30]}  issued={m["issue_date"].date() if pd.notna(m["issue_date"]) else None}')
            else:
                print(f'  {c}')
else:
    print(f'Arthur price.csv not found at {arthur_path} — skipping comparison')


Shared CUSIPs:      107
Only in this NB:    23
Only in Arthur:     41

First 20 CUSIPs we keep that Arthur drops:
  00138EAY0  Corebridge Global Funding  issued=2024-06-24
  00138EBB9  Corebridge Global Funding  issued=2024-08-22
  13607PVU5  Canadian Imperial Bank of Comm  issued=2025-01-14
  14913UAQ3  Caterpillar Financial Services  issued=2024-08-16
  14913UAS9  Caterpillar Financial Services  issued=2024-11-15
  166756AZ9  Chevron USA Inc  issued=2025-02-26
  37045XEX0  General Motors Financial Co In  issued=2024-06-18
  459058JR5  International Bank for Reconst  issued=2002-10-21
  57629XDA3  MassMutual Global Funding II  issued=2024-09-17
  64952XFJ5  New York Life Global Funding  issued=2024-12-13
  66815M2R7  Northwestern Mutual Global Fun  issued=2024-03-25
  66815M2S5  Northwestern Mutual Global Fun  issued=2024-05-28
  69371RT55  PACCAR Financial Corp  issued=2024-11-25
  86562MDT4  Sumitomo Mitsui Financial Grou  issued=2025-01-15
  89115A3G5  Toronto-Dominion Bank/The  is

## 6. US Treasury Curve from FRED & Credit Spread (G-Spread)

G-Spread = Bond YTM - Interpolated Treasury yield at matching remaining maturity

In [91]:
from scipy.optimize import brentq

# Download US Treasury Constant Maturity rates from FRED
FRED_SERIES = {
    'DGS1MO': 1/12, 'DGS3MO': 3/12, 'DGS6MO': 6/12,
    'DGS1': 1, 'DGS2': 2, 'DGS3': 3, 'DGS5': 5,
    'DGS7': 7, 'DGS10': 10, 'DGS20': 20, 'DGS30': 30,
}

start = str(mid.index.min().date())
end   = str(mid.index.max().date())

treasury_dfs = {}
for series_id, tenor in FRED_SERIES.items():
    url = f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={series_id}&cosd={start}&coed={end}'
    try:
        df = pd.read_csv(url, parse_dates=['observation_date'], index_col='observation_date', na_values='.')
        df.columns = [tenor]
        treasury_dfs[tenor] = df
        print(f'  {series_id} ({tenor}y): {df.dropna().shape[0]} obs')
    except Exception as e:
        print(f'  FAILED {series_id}: {e}')

# Combine and forward-fill (weekends/holidays)
treasury = pd.concat(treasury_dfs.values(), axis=1).sort_index()
treasury = treasury.loc[start:end].ffill()

# Convert from percent to decimal for YTM comparison
treasury_decimal = treasury / 100

print(f'\nTreasury curve: {treasury.shape}')
print(f'Tenors available: {sorted(treasury.columns.tolist())}')
treasury.tail(3)


  DGS1MO (0.08333333333333333y): 496 obs
  DGS3MO (0.25y): 496 obs
  DGS6MO (0.5y): 496 obs
  DGS1 (1y): 496 obs
  DGS2 (2y): 496 obs
  DGS3 (3y): 496 obs
  DGS5 (5y): 496 obs
  DGS7 (7y): 496 obs
  DGS10 (10y): 496 obs
  DGS20 (20y): 496 obs
  DGS30 (30y): 496 obs

Treasury curve: (520, 11)
Tenors available: [0.08333333333333333, 0.25, 0.5, 1.0, 2.0, 3.0, 5.0, 7.0, 10.0, 20.0, 30.0]


,0.083333,0.250000,0.500000,1.000000,2.000000,3.000000,5.000000,7.000000,10.000000,20.000000,30.000000
observation_date,,,,,,,,,,,
2026-02-24,3.72,3.69,3.62,3.52,3.43,3.47,3.61,3.81,4.04,4.63,4.70
2026-02-25,3.71,3.69,3.62,3.53,3.45,3.49,3.61,3.82,4.05,4.63,4.70
2026-02-26,3.74,3.68,3.61,3.52,3.42,3.46,3.57,3.78,4.02,4.60,4.67


In [92]:
# Build bond metadata lookup from bv2 (coupon, frequency, maturity)
cusips = list(mid.columns)

cusip_meta = {}
for _, row in bv2.iterrows():
    c = row['CUSIP']
    if c in cusips:
        cusip_meta[c] = {
            'coupon':   row['Cpn'],
            'freq':     int(row['CPN_FREQ']) if pd.notna(row.get('CPN_FREQ')) else 2,
            'maturity': row['maturity'],
        }

print(f'Metadata matched for {len(cusip_meta)} of {len(cusips)} bonds')

# --- YTM solver: proper cash-flow schedule, BEY convention ---

def generate_coupon_dates(maturity_date, freq, valuation_date):
    """Generate future coupon dates by working backwards from maturity."""
    interval_months = 12 // freq
    dates = []
    d = pd.Timestamp(maturity_date)
    while d > pd.Timestamp(valuation_date):
        dates.append(d)
        d = d - pd.DateOffset(months=interval_months)
    dates.sort()
    return dates

def compute_ytm(price, coupon_rate, freq, maturity_date, valuation_date, face=100):
    """
    Compute YTM using actual cash flow dates and semi-annual compounding (BEY).
    Directly comparable to US Treasury yields from FRED.
    """
    if pd.isna(price) or price <= 0 or pd.isna(maturity_date):
        return np.nan
    mat_dt = pd.Timestamp(maturity_date)
    val_dt = pd.Timestamp(valuation_date)
    if mat_dt <= val_dt:
        return np.nan
    cpn_dates = generate_coupon_dates(mat_dt, freq, val_dt)
    if len(cpn_dates) == 0:
        return np.nan
    cpn_payment = face * coupon_rate / 100 / freq
    times_yr = [(d - val_dt).days / 365.25 for d in cpn_dates]
    cfs = [cpn_payment] * len(cpn_dates)
    cfs[-1] += face
    def pv_diff(y):
        return sum(cf / (1 + y / 2) ** (2 * t) for cf, t in zip(cfs, times_yr)) - price
    try:
        return brentq(pv_diff, -0.05, 1.0, xtol=1e-8)
    except (ValueError, RuntimeError):
        return np.nan

tenors_sorted = sorted(FRED_SERIES.values())

def interp_treasury(row, years_to_mat):
    """Linearly interpolate the Treasury curve for a given maturity."""
    if years_to_mat <= 0:
        return np.nan
    y = max(tenors_sorted[0], min(tenors_sorted[-1], years_to_mat))
    vals = [row.get(t, np.nan) for t in tenors_sorted]
    valid = [(t, v) for t, v in zip(tenors_sorted, vals) if not np.isnan(v)]
    if len(valid) < 2:
        return np.nan
    ts, vs = zip(*valid)
    return np.interp(y, ts, vs)

# Sanity check
test_ytm = compute_ytm(100, 5.0, 2, pd.Timestamp('2029-03-01'), pd.Timestamp('2024-03-01'))
print(f'Sanity check: par bond 5% coupon => YTM = {test_ytm*100:.3f}% (should be ~5.0%)')
print('YTM solver ready.')


Metadata matched for 130 of 130 bonds
Sanity check: par bond 5% coupon => YTM = 5.000% (should be ~5.0%)
YTM solver ready.


In [93]:
# Compute G-Spread for each bond on each date
# G-Spread = Bond YTM (BEY) - Interpolated Treasury yield (BEY) at matching remaining maturity, in bps

treasury_reindexed = treasury_decimal.reindex(mid.index).ffill()

credit_spread = pd.DataFrame(index=mid.index)
credit_spread.index.name = 'Date'

n_bonds = len(cusips)
for i, cusip in enumerate(cusips):
    if (i + 1) % 25 == 0 or i == 0:
        print(f'  Processing bond {i+1}/{n_bonds}...')

    meta = cusip_meta.get(cusip)
    if meta is None:
        credit_spread[cusip] = np.nan
        continue

    cpn      = meta['coupon']
    freq     = meta['freq']
    mat_date = meta['maturity']

    spreads = []
    for date in mid.index:
        price = mid.at[date, cusip]

        if pd.isna(price) or pd.isna(mat_date):
            spreads.append(np.nan)
            continue

        years_to_mat = (pd.Timestamp(mat_date) - pd.Timestamp(date)).days / 365.25
        if years_to_mat <= 0:
            spreads.append(np.nan)
            continue

        ytm = compute_ytm(price, cpn, freq, mat_date, date)

        if date in treasury_reindexed.index:
            tsy_yield = interp_treasury(treasury_reindexed.loc[date], years_to_mat)
        else:
            tsy_yield = np.nan

        if np.isnan(ytm) or np.isnan(tsy_yield):
            spreads.append(np.nan)
        else:
            spreads.append((ytm - tsy_yield) * 10_000)  # bps

    credit_spread[cusip] = spreads

print(f'\nCredit Spread (G-Spread) shape: {credit_spread.shape}')

neg_count = (credit_spread[cusips] < 0).sum()
neg_bonds = neg_count[neg_count > 0]
print(f'Bonds with any negative G-spread: {len(neg_bonds)}')
if len(neg_bonds):
    print('  (May indicate bonds trading tighter than Treasuries — e.g., agency/supranational)')
    print(neg_bonds.sort_values(ascending=False).head(10))

print(f'\nOverall mean spread: {credit_spread[cusips].mean().mean():.1f} bps')
print(f'Median spread:       {credit_spread[cusips].median().median():.1f} bps')


  Processing bond 1/130...
  Processing bond 25/130...
  Processing bond 50/130...
  Processing bond 75/130...
  Processing bond 100/130...
  Processing bond 125/130...

Credit Spread (G-Spread) shape: (499, 130)


/var/folders/zg/9jdfnwbd0v56xz8n032gjqtc0000gn/T/ipykernel_23878/3023922234.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  credit_spread[cusip] = spreads
/var/folders/zg/9jdfnwbd0v56xz8n032gjqtc0000gn/T/ipykernel_23878/3023922234.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  credit_spread[cusip] = spreads
/var/folders/zg/9jdfnwbd0v56xz8n032gjqtc0000gn/T/ipykernel_23878/3023922234.py:48: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poo

Bonds with any negative G-spread: 14
  (May indicate bonds trading tighter than Treasuries — e.g., agency/supranational)
BO1649512    26
459058JR5    20
BW4937361     4
191216DD9     2
66815M2R7     2
022249AU0     1
13607PVU5     1
14913UAQ3     1
24422EWL9     1
37045XEX0     1
dtype: int64

Overall mean spread: 110.4 bps
Median spread:       106.7 bps


## 6b. Credit Spread using Dirty Price (Clean + Accrued Interest)

The sawtooth pattern in G-spreads comes from using clean prices. Bloomberg quotes clean prices, but YTM should be computed from dirty price = clean price + accrued interest.

In [94]:
def compute_accrued_interest(coupon_rate, freq, maturity_date, valuation_date, face=100):
    """
    Accrued interest = coupon_payment * (days since last coupon / days in coupon period).
    Uses actual/actual day count convention.
    """
    mat_dt = pd.Timestamp(maturity_date)
    val_dt = pd.Timestamp(valuation_date)
    interval_months = 12 // freq
    d = mat_dt
    prev_cpn = None
    next_cpn = None
    while d > val_dt:
        next_cpn = d
        d = d - pd.DateOffset(months=interval_months)
    prev_cpn = d
    if prev_cpn is None or next_cpn is None:
        return 0.0
    days_since  = (val_dt - prev_cpn).days
    days_in_period = (next_cpn - prev_cpn).days
    if days_in_period <= 0:
        return 0.0
    coupon_payment = face * coupon_rate / 100 / freq
    return coupon_payment * (days_since / days_in_period)

# Sanity test
ai_test = compute_accrued_interest(5.0, 2, '2029-03-01', '2024-06-01')
print(f'Sanity check: 5% semi-annual, 3 months after coupon => AI = {ai_test:.4f} (expect ~1.25)')

# Compute G-Spread using dirty price = clean price + accrued interest
credit_spread_dirty = pd.DataFrame(index=mid.index)
credit_spread_dirty.index.name = 'Date'

for i, cusip in enumerate(cusips):
    if (i + 1) % 25 == 0 or i == 0:
        print(f'  Processing bond {i+1}/{n_bonds}...')

    meta = cusip_meta.get(cusip)
    if meta is None:
        credit_spread_dirty[cusip] = np.nan
        continue

    cpn      = meta['coupon']
    freq     = meta['freq']
    mat_date = meta['maturity']

    spreads = []
    for date in mid.index:
        clean_price = mid.at[date, cusip]

        if pd.isna(clean_price) or pd.isna(mat_date):
            spreads.append(np.nan)
            continue

        years_to_mat = (pd.Timestamp(mat_date) - pd.Timestamp(date)).days / 365.25
        if years_to_mat <= 0:
            spreads.append(np.nan)
            continue

        ai          = compute_accrued_interest(cpn, freq, mat_date, date)
        dirty_price = clean_price + ai
        ytm         = compute_ytm(dirty_price, cpn, freq, mat_date, date)

        if date in treasury_reindexed.index:
            tsy_yield = interp_treasury(treasury_reindexed.loc[date], years_to_mat)
        else:
            tsy_yield = np.nan

        if np.isnan(ytm) or np.isnan(tsy_yield):
            spreads.append(np.nan)
        else:
            spreads.append((ytm - tsy_yield) * 10_000)  # bps

    credit_spread_dirty[cusip] = spreads

print(f'\nCredit Spread (Dirty Price) shape: {credit_spread_dirty.shape}')
neg_clean = (credit_spread[cusips] < 0).sum().sum()
neg_dirty = (credit_spread_dirty[cusips] < 0).sum().sum()
print(f'Negative spread observations: Clean={neg_clean}, Dirty={neg_dirty}')
print(f'Mean spread: Clean={credit_spread[cusips].mean().mean():.1f} bps, '
      f'Dirty={credit_spread_dirty[cusips].mean().mean():.1f} bps')


Sanity check: 5% semi-annual, 3 months after coupon => AI = 1.2500 (expect ~1.25)
  Processing bond 1/130...
  Processing bond 25/130...
  Processing bond 50/130...
  Processing bond 75/130...
  Processing bond 100/130...
  Processing bond 125/130...

Credit Spread (Dirty Price) shape: (499, 130)
Negative spread observations: Clean=63, Dirty=289
Mean spread: Clean=110.4 bps, Dirty=61.7 bps


/var/folders/zg/9jdfnwbd0v56xz8n032gjqtc0000gn/T/ipykernel_23878/55860904.py:73: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  credit_spread_dirty[cusip] = spreads
/var/folders/zg/9jdfnwbd0v56xz8n032gjqtc0000gn/T/ipykernel_23878/55860904.py:73: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  credit_spread_dirty[cusip] = spreads
/var/folders/zg/9jdfnwbd0v56xz8n032gjqtc0000gn/T/ipykernel_23878/55860904.py:73: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which h

## 7. Export to Excel

In [95]:
OUTPUT = Path('bloomberg_first_batch_cleaned.xlsx')

with pd.ExcelWriter(OUTPUT, engine='openpyxl') as writer:
    mid.reset_index().to_excel(writer, sheet_name='Mid_Clean', index=False)
    bid.reset_index().to_excel(writer, sheet_name='Bid_Clean', index=False)
    ask.reset_index().to_excel(writer, sheet_name='Ask_Clean', index=False)
    credit_spread.reset_index().to_excel(writer, sheet_name='Credit_Spread_bps', index=False)
    credit_spread_dirty.reset_index().to_excel(writer, sheet_name='Credit_Spread_Dirty', index=False)
    treasury.reset_index().to_excel(writer, sheet_name='Treasury_Curve', index=False)

print(f'Exported to {OUTPUT}')
print(f'\nSheets:')
print(f'  Mid_Clean            - Cleaned mid prices {mid.shape}')
print(f'  Bid_Clean            - Cleaned bid prices {bid.shape}')
print(f'  Ask_Clean            - Cleaned ask prices {ask.shape}')
print(f'  Credit_Spread_bps    - G-Spread from clean price in bps {credit_spread.shape}')
print(f'  Credit_Spread_Dirty  - G-Spread from dirty price in bps {credit_spread_dirty.shape}')
print(f'  Treasury_Curve       - FRED Treasury rates used {treasury.shape}')
print(f'\nAll tabs: {len(cusips)} bonds x {len(mid)} trading days')


Exported to bloomberg_first_batch_cleaned.xlsx

Sheets:
  Mid_Clean            - Cleaned mid prices (499, 130)
  Bid_Clean            - Cleaned bid prices (499, 130)
  Ask_Clean            - Cleaned ask prices (499, 130)
  Credit_Spread_bps    - G-Spread from clean price in bps (499, 130)
  Credit_Spread_Dirty  - G-Spread from dirty price in bps (499, 130)
  Treasury_Curve       - FRED Treasury rates used (520, 11)

All tabs: 130 bonds x 499 trading days
